# Rehavision デモ Notebook

人工知能システム開発 第7班「Rehavision」

Google Colab上でGemini APIを使い、Prompt Template + Reference（RAG風の外部情報）に基づいて、
作業療法士・家族からの質問にAIが回答し、TTSで読み上げるデモです。

課題PDF「デモ」スライドで示されている、複数モデルへのフォールバック方式（クォータ超過・廃止モデル対策）を実装しています。

## 使い方
1. Colabメニューの鍵アイコン（Secrets）から `GOOGLE_API_KEY` を登録し、Notebook access をONにする
2. 上から順にセルを実行する
3. 「質問」セルの `question` を書き換えて再実行すると、任意の質問で試せる

## 1. セットアップ

In [ ]:
!pip install -q google-generativeai gTTS

In [ ]:
import os
import google.generativeai as genai
from gtts import gTTS
from IPython.display import Audio, display

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except ImportError:
    api_key = os.environ.get("GOOGLE_API_KEY")

assert api_key, "GOOGLE_API_KEY が見つかりません。Colab Secretsまたは環境変数に設定してください。"
genai.configure(api_key=api_key)
print("Gemini API 設定完了")

## 2. モデルのフォールバック呼び出し
無料枠のクォータ超過（429）やモデル廃止（404）に対応するため、候補モデルを順に試す。

In [ ]:
MODEL_CANDIDATES = [
    "models/gemini-2.5-flash",
    "models/gemini-2.5-pro",
    "models/gemini-2.0-flash",
    "models/gemini-2.0-flash-001",
    "models/gemini-2.0-flash-lite-001",
    "models/gemini-2.0-flash-lite",
    "models/gemini-flash-latest",
]


def ask_custom_llm(prompt, model_candidates=MODEL_CANDIDATES, verbose=True):
    last_error = None
    for model_name in model_candidates:
        if verbose:
            print(f"-> {model_name} で実行を試みます...")
        try:
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(prompt)
            if verbose:
                print(f"\u2705 成功 ({model_name})")
            return response.text.strip()
        except Exception as exc:
            last_error = exc
            if verbose:
                print(f"[失敗] {model_name}: {exc}")
            continue
    raise RuntimeError(f"すべての候補モデルで失敗しました: {last_error}")

## 3. Prompt Template / Reference の読み込み

リポジトリ（`rehavision/prompts/07_prompt.txt`, `07_reference.txt`）をColabにアップロードまたはgit cloneしている場合は
ファイルから読み込む。単体で使う場合は、下のインラインの文字列にフォールバックする。

In [ ]:
PROMPT_TEMPLATE_PATH = "../prompts/07_prompt.txt"
REFERENCE_PATH = "../prompts/07_reference.txt"

INLINE_PROMPT_TEMPLATE = """# 回答条件
- あなたは「Rehavision」の音声アシスタントです。作業療法士または患者のご家族からの質問に、下記「患者情報」に基づいて答えてください。
- 医療専門用語を使う場合は、家族にも分かる簡単な言葉で補足してください。
- 「患者情報」に記載がない内容は、推測で断定せず「記録には該当する情報がありません。担当の作業療法士にご確認ください。」と回答してください。
- 診断や治療方針の変更につながるような断定的な発言は避けてください。
- 回答は3文以内、日本語の自然な話し言葉でまとめてください。

# フォーマット
- 箇条書きや記号(・, -, *, #)は使わないでください。音声合成(TTS)でそのまま読み上げるため、話し言葉の文章のみを出力してください。
- 文末は「です」「ます」調で統一してください。

# 患者情報
{reference_str}

# 質問
質問：{question}
"""

INLINE_REFERENCE = """■ 患者情報
氏名：田中 太郎（仮名）
年齢：78歳
主病名：脳梗塞後遗症による右片麻痺
既往歴：2型糖尿病

■ リハビリ方针（担当作業療法士：高原 光月）
目標：屋内歩行の自立
現在のフェーズ：Rehavisionシステム（天井カメラ・ LiDAR・床面プロジェクション）を用いた歩行訓練

■ 直近の訓練記録
2026-08-01：歩行訓練を実施。目標5歩に対し5歩を達成。序盤は左足の踏み出しがやや不安定だったが、訓練後半にかけてスムーズさが改善した。次回の目標歩数を7歩に設定。
2026-07-28：歩行訓練を実施。疲労のため５歩中３歩で中断。無理のない範囲で継続する方針。
2026-07-24：初回評価。バランス能力・筋力の測定を実施し、訓練プログラムを作成。

■ 生活・食事に関する注意事項
2型糖尿病のため、間食や糖分の多い飲食物は控える必要があります。コーヒー自体は問題ありませんが、砂糖を入れることは避けてください。

■ 家族への伝達事項
転倒リスクがあるため、歩行時は目を離さず見守ってください。ご本人のペースを尊重し、焦らせないようにお願いします。
"""

try:
    with open(PROMPT_TEMPLATE_PATH, encoding="utf-8") as f:
        prompt_template = f.read()
    with open(REFERENCE_PATH, encoding="utf-8") as f:
        reference_str = f.read()
    print("prompts/ ディレクトリから読み込みました")
except FileNotFoundError:
    prompt_template = INLINE_PROMPT_TEMPLATE
    reference_str = INLINE_REFERENCE
    print("ファイルが見つからないため、インラインのテンプレートを使用します")

## 4. 質問応答デモ
審査員役の質問をここに入力して実行する。

In [ ]:
question = "田中さんはコーヒー飲めますか？"

prompt = prompt_template.format(reference_str=reference_str, question=question)
print("=== LLMの動作テスト ===")
print(f"ユーザー：{question}")
answer = ask_custom_llm(prompt)
print(f"AIの回答：{answer}\n")

## 5. TTSで読み上げ

In [ ]:
tts = gTTS(text=answer, lang="ja")
tts.save("answer.mp3")
display(Audio("answer.mp3", autoplay=False))

## 6. 複数質問での一括テスト（WOZコーパス想定）
課題資料のWOZ対話例（田中さんのバイタル表示、コーヒーの可否など）を想定した質問リストで動作確認する。

In [ ]:
test_questions = [
    "田中さんの今日の様子を教えてください",
    "田中さんはコーヒー飲めますか？",
    "歩行訓練の進み具合はどうですか？",
    "次回の目標は何ですか？",
    "田中さんは今日何を食べましたか？",  # reference に情報がない質問（ハルシネーション対策の確認用）
]

print("=== LLMの動作テスト ===")
for i, q in enumerate(test_questions):
    print(f"ユーザー：{q}")
    a = ask_custom_llm(prompt_template.format(reference_str=reference_str, question=q), verbose=False)
    print(f"AIの回答：{a}\n")